# IT4060 - Machine Learning
## Lab 4 - Optimizing Linear Regression
### Part A - Demonstration: House-Price Regression

This notebook is used during the laboratory session to demonstrate feature scaling, the cost function, gradient descent, learning-rate behaviour, the normal equation, scikit-learn verification, and Ridge regularization.

**No submission is required for Part A.**

## Learning focus
This lab is about **how the parameters of linear regression are obtained**: feature scaling, cost minimization, gradient descent, learning rate, convergence, the normal equation, and Ridge regularization.

---
# Part A - Inclass Walkthrough: House-Price Regression
We use two input features: floor area and number of bedrooms. The target is house price.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True, precision=4)


### A1. Load the demonstration data
The data are embedded in the notebook so no separate file is required.

In [ ]:
house_data = np.array([
[2104,3,399900],[1600,3,329900],[2400,3,369000],[1416,2,232000],
[3000,4,539900],[1985,4,299900],[1534,3,314900],[1427,3,198999],
[1380,3,212000],[1494,3,242500],[1940,4,239999],[2000,3,347000],
[1890,3,329999],[4478,5,699900],[1268,3,259900],[2300,4,449900],
[1320,2,299900],[1236,3,199900],[2609,4,499998],[3031,4,599000],
[1767,3,252900],[1888,2,255000],[1604,3,242900],[1962,4,259900],
[3890,3,573900],[1100,3,249900],[1458,3,464500],[2526,3,469000],
[2200,3,475000],[2637,3,299900],[1839,2,349900],[1000,1,169900],
[2040,4,314900],[3137,3,579900],[1811,4,285900],[1437,3,249900],
[1239,3,229900],[2132,4,345000],[4215,4,549000],[2162,4,287000],
[1664,2,368500],[2238,3,329900],[2567,4,314000],[1200,3,299000],
[852,2,179900],[1852,4,299900],[1203,3,239500]
], dtype=float)

house_df = pd.DataFrame(house_data, columns=['floor_area','bedrooms','price'])
house_df.head()


In [ ]:
print(house_df[['floor_area','bedrooms']].agg(['min','max','mean','std']))


### A2. Matrix form
For two features, the model is $\hat y=\theta_0+\theta_1x_1+\theta_2x_2$. For all observations together: $\hat{\mathbf y}=X\theta$.

In [ ]:
X_raw = house_df[['floor_area','bedrooms']].to_numpy(dtype=float)
y = house_df['price'].to_numpy(dtype=float).reshape(-1,1)

mu = X_raw.mean(axis=0)
sigma = X_raw.std(axis=0)
X_scaled = (X_raw - mu) / sigma

X = np.c_[np.ones((len(X_scaled),1)), X_scaled]
theta0 = np.zeros((X.shape[1],1))

print('X shape:', X.shape)
print('theta shape:', theta0.shape)
print('Feature means after scaling:', X_scaled.mean(axis=0))
print('Feature std after scaling:', X_scaled.std(axis=0))


### A3. Cost function
$J(\theta)=\frac{1}{2m}(X\theta-y)^T(X\theta-y)$

In [ ]:
def compute_cost(X, y, theta):
    m = len(y)
    error = X @ theta - y
    return float(((error.T @ error) / (2*m)).item())

print('Initial cost:', compute_cost(X, y, theta0))


### A4. Gradient descent
$\theta \leftarrow \theta - \frac{\alpha}{m}X^T(X\theta-y)$

In [ ]:
def gradient_descent(X, y, theta, alpha=0.1, iterations=500, stop_on_nonfinite=True):
    m = len(y)
    history = []
    theta = theta.astype(float).copy()

    for i in range(iterations):
        error = X @ theta - y
        gradient = (X.T @ error) / m
        theta = theta - alpha * gradient
        cost = compute_cost(X, y, theta)
        history.append(cost)

        if stop_on_nonfinite and not np.isfinite(cost):
            print(f'Stopped at iteration {i+1}: cost became non-finite.')
            break

    return theta, np.asarray(history)

alpha = 0.1
theta_gd, history = gradient_descent(X, y, theta0, alpha=alpha, iterations=500)
print('Final theta:', theta_gd, sep='\n')
print('Final cost:', history[-1])


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(history)
plt.xlabel('Iteration')
plt.ylabel('Cost J(theta)')
plt.title('Gradient Descent Convergence')
plt.grid(alpha=0.25)
plt.show()


### A5. Learning-rate experiment

In [ ]:
rates = [0.01, 0.1, 0.8, 1.1]
plt.figure(figsize=(7,4.5))

for a in rates:
    _, h = gradient_descent(X, y, theta0, alpha=a, iterations=150)
    plt.plot(h, label=f'alpha={a}')

plt.xlabel('Iteration')
plt.ylabel('Cost J(theta)')
plt.title('Effect of Learning Rate')
plt.yscale('log')
plt.legend()
plt.grid(alpha=0.25)
plt.show()


### A6. What happens without feature scaling?
Here we deliberately try the same basic optimizer on raw feature values. A learning rate that works on standardized data can produce very large updates on the raw floor-area feature.

In [ ]:
X_unscaled = np.c_[np.ones((len(X_raw),1)), X_raw]
theta_unscaled0 = np.zeros((X_unscaled.shape[1],1))

_, h_unscaled = gradient_descent(
    X_unscaled, y, theta_unscaled0,
    alpha=0.1, iterations=20, stop_on_nonfinite=True
)

print('First few unscaled costs:', h_unscaled[:5])


### A7. Predict a 1650 sq. ft., 3-bedroom house

In [ ]:
new_house = np.array([[1650.0, 3.0]])
new_house_scaled = (new_house - mu) / sigma
new_house_X = np.c_[np.ones((1,1)), new_house_scaled]

pred_gd = float((new_house_X @ theta_gd).item())
print('Gradient-descent prediction:', pred_gd)


### A8. Normal equation

In [ ]:
theta_ne = np.linalg.pinv(X.T @ X) @ X.T @ y
pred_ne = float((new_house_X @ theta_ne).item())

print('Normal-equation theta:', theta_ne, sep='\n')
print('Normal-equation prediction:', pred_ne)


### A9. Verify with scikit-learn

In [ ]:
from sklearn.linear_model import LinearRegression

lin = LinearRegression()
lin.fit(X_scaled, y.ravel())
pred_sk = float(lin.predict(new_house_scaled)[0])

print('Gradient descent :', pred_gd)
print('Normal equation  :', pred_ne)
print('scikit-learn     :', pred_sk)


### A10. Ridge regularization
Ridge adds an L2 penalty that discourages very large coefficients. In scikit-learn, the parameter is called `alpha`; in the lab sheet we refer to the regularization strength as $\lambda$ to avoid confusing it with the gradient-descent learning rate.

In [ ]:
from sklearn.linear_model import Ridge

ridge_rows = []
for lam in [0.01, 1.0, 100.0, 10000.0]:
    model = Ridge(alpha=lam)
    model.fit(X_scaled, y.ravel())
    coef_norm = np.linalg.norm(model.coef_)
    pred = float(model.predict(new_house_scaled)[0])
    ridge_rows.append([lam, *model.coef_, coef_norm, pred])

ridge_demo = pd.DataFrame(
    ridge_rows,
    columns=['lambda','coef_floor_area','coef_bedrooms','coefficient_L2_norm','prediction_1650_3']
)
ridge_demo
